# Week 5 – Day 4
# CrewAI: Multi-Agent Collaboration, Roles & Task Delegation

# Task 1 – Multi-Agent Design Thinking

For this task, we design a small CrewAI team capable of completing a realistic remote sensing data analysis workflow.

### Selected Business Task

**Environmental Monitoring Report Generation**

An environmental consulting organization has received a CSV dataset exported from a GIS/remote sensing workflow. The dataset contains vegetation indices (NDVI), land cover information, rainfall measurements, and other environmental observations collected from multiple monitoring sites.

The objective is to transform the raw dataset into a stakeholder-ready environmental assessment report by performing the following steps:

1. Inspect and validate the dataset for missing or inconsistent observations.
2. Analyze the cleaned data to identify vegetation health patterns and environmental trends.
3. Produce a professional environmental monitoring report for decision-makers.

This workflow naturally separates into specialized responsibilities, making it an ideal candidate for a multi-agent system.

## Agent Design
<center>
<img src="environmental_monitoring_crew.png" width="900">
</center>
The environmental monitoring workflow is divided among three specialized agents. Each agent has a clearly defined responsibility with minimal overlap.

| Agent | Role | Primary Responsibility |
|--------|------|------------------------|
| Agent 1 | Environmental Data Quality Specialist | Validate and prepare the remote sensing dataset for analysis |
| Agent 2 | Remote Sensing Analyst | Analyze vegetation health, land cover, and environmental trends |
| Agent 3 | Environmental Assessment Report Writer | Produce a stakeholder-ready environmental report based on the analysis |

This separation follows the principle of specialization, allowing each agent to focus on a single well-defined task.


In [3]:
agent_design = {
    "Environmental Data Quality Specialist": {
        "Goal": (
            "Inspect, validate, and prepare environmental monitoring datasets "
            "for reliable analysis."
        ),
        "Backstory": (
            "An experienced environmental data specialist who identifies "
            "missing values, inconsistent records, abnormal sensor readings, "
            "and data quality issues before scientific analysis begins."
        )
    },

    "Remote Sensing Analyst": {
        "Goal": (
            "Analyze cleaned remote sensing data to identify vegetation "
            "health patterns and environmental trends."
        ),
        "Backstory": (
            "A remote sensing analyst experienced in interpreting NDVI, "
            "land cover classifications, rainfall observations, and "
            "environmental indicators to generate actionable insights."
        )
    },

    "Environmental Assessment Report Writer": {
        "Goal": (
            "Transform technical environmental analyses into clear, "
            "professional reports for decision-makers."
        ),
        "Backstory": (
            "An environmental reporting specialist who communicates complex "
            "scientific findings in a concise and accessible manner for "
            "government agencies, environmental consultants, and stakeholders."
        )
    }
}

for role, info in agent_design.items():
    print(f"\n{'='*70}")
    print(f"ROLE: {role}")
    print(f"Goal: {info['Goal']}")
    print(f"Backstory: {info['Backstory']}")


ROLE: Environmental Data Quality Specialist
Goal: Inspect, validate, and prepare environmental monitoring datasets for reliable analysis.
Backstory: An experienced environmental data specialist who identifies missing values, inconsistent records, abnormal sensor readings, and data quality issues before scientific analysis begins.

ROLE: Remote Sensing Analyst
Goal: Analyze cleaned remote sensing data to identify vegetation health patterns and environmental trends.
Backstory: A remote sensing analyst experienced in interpreting NDVI, land cover classifications, rainfall observations, and environmental indicators to generate actionable insights.

ROLE: Environmental Assessment Report Writer
Goal: Transform technical environmental analyses into clear, professional reports for decision-makers.
Backstory: An environmental reporting specialist who communicates complex scientific findings in a concise and accessible manner for government agencies, environmental consultants, and stakeholders.

## Why Multiple Specialized Agents?

Environmental data analysis involves multiple stages that require different areas of expertise, from validating raw observations to interpreting environmental indicators and communicating findings to stakeholders. Assigning these responsibilities to specialized agents improves task focus, produces more structured intermediate outputs, and enables each stage of the workflow to build on the previous one.

However, for small datasets or straightforward reporting tasks, a single well-designed agent may be sufficient. In such cases, the additional coordination overhead of a multi-agent system may not justify the increased complexity or token usage.

# Task 2 – Build Agents & Assign Tools

In this task, we implement the three specialized CrewAI agents designed in Task 1.

Each agent receives:
- its own LLM configuration,
- only the tools required for its responsibility,
- a role, goal, and backstory that reflect its specialization.

Following the principle of least privilege, each agent is provided only with the tools necessary for its assigned responsibility. This encourages specialization, reduces unnecessary tool usage, and creates a more realistic multi-agent workflow.

In [4]:
# Imports
from crewai import Agent, LLM
from crewai.tools import tool

from dotenv import load_dotenv
import os
import pandas as pd

In [5]:
# Environment and LLM
load_dotenv()

API_KEY = os.getenv("NETIXSOL_API_KEY")
BASE_URL = "https://llm.netixsol.com/v1"

# CrewAI / LiteLLM compatibility
os.environ["OPENAI_API_KEY"] = API_KEY
os.environ["OPENAI_BASE_URL"] = BASE_URL

MODEL = "batch"

llm = LLM(
    model=f"openai/{MODEL}",
    api_key=API_KEY,
    base_url="https://llm.netixsol.com/v1",
    temperature=0
)

## Tool Design

The environmental monitoring workflow requires different capabilities at different stages.

Rather than giving every agent access to every available tool, each agent receives only the tools needed to perform its specific responsibility.

This mirrors how real environmental and GIS teams distribute responsibilities among specialists.

In [6]:
# Tools

#tool # 1
@tool
def dataset_quality_report(file_path: str) -> str:
    """
    Generate a data quality report for an environmental monitoring dataset.
    """
    df = pd.read_csv(file_path)

    report = []

    report.append("=== DATASET OVERVIEW ===")
    report.append(f"Rows: {df.shape[0]}")
    report.append(f"Columns: {df.shape[1]}")
    report.append(f"\nColumns: {list(df.columns)}")

    report.append("\n=== MISSING VALUES ===")
    report.append(df.isnull().sum().to_string())

    report.append(f"\nDuplicate Rows: {df.duplicated().sum()}")

    # NDVI validation
    if "NDVI" in df.columns:
        invalid_ndvi = df[(df["NDVI"] < -1) | (df["NDVI"] > 1)]
        report.append(f"\nInvalid NDVI Values: {len(invalid_ndvi)}")

    # Rainfall validation
    if "Rainfall_mm" in df.columns:
        negative_rainfall = df[df["Rainfall_mm"] < 0]
        report.append(f"Negative Rainfall Values: {len(negative_rainfall)}")

    return "\n".join(report)

In [7]:
# tool # 2
@tool
def environmental_statistics(file_path: str) -> str:
    """
    Generate descriptive statistics for an environmental monitoring dataset.
    """
    df = pd.read_csv(file_path)

    report = []

    report.append("=== NUMERICAL SUMMARY ===")
    report.append(df.describe().to_string())

    if "Land_Cover" in df.columns:
        report.append("\n\n=== LAND COVER DISTRIBUTION ===")
        report.append(df["Land_Cover"].value_counts().to_string())

    if "Vegetation_Health" in df.columns:
        report.append("\n\n=== VEGETATION HEALTH ===")
        report.append(df["Vegetation_Health"].value_counts().to_string())

    return "\n".join(report)

In [8]:
# Agent 1

environmental_data_quality_specialist = Agent(
    role="Environmental Data Quality Specialist",

    goal=(
        "Inspect, validate, and prepare environmental monitoring datasets "
        "for reliable analysis."
    ),

    backstory=(
        "You are an experienced environmental data specialist responsible "
        "for identifying missing values, inconsistent records, abnormal "
        "sensor observations, and other data quality issues before analysis."
    ),

    llm=llm,

    tools=[dataset_quality_report],

    verbose=True
)

In [9]:
# Agent 2
remote_sensing_analyst = Agent(
    role="Remote Sensing Analyst",

    goal=(
        "Analyze cleaned environmental datasets to identify vegetation "
        "health patterns, land cover trends, and environmental insights."
    ),

    backstory=(
        "You are an experienced remote sensing analyst who specializes in "
        "interpreting NDVI values, rainfall observations, land cover "
        "information, and other environmental indicators."
    ),

    llm=llm,

    tools=[environmental_statistics],

    verbose=True
)

In [10]:
# Agent 3

environmental_report_writer = Agent(
    role="Environmental Assessment Report Writer",

    goal=(
        "Produce clear, professional environmental assessment reports "
        "for decision-makers and stakeholders."
    ),

    backstory=(
        "You are an environmental reporting specialist who transforms "
        "technical environmental analyses into concise, well-structured "
        "reports suitable for policymakers, researchers, and environmental "
        "consultants."
    ),

    llm=llm,

    tools=[],

    verbose=True
)

## Tool Assignment Justification

| Agent | Assigned Tool(s) | Justification |
|--------|------------------|---------------|
| Environmental Data Quality Specialist | `dataset_quality_report` | Generates a comprehensive quality assessment, including missing values, duplicate records, and invalid environmental measurements before analysis begins. |
| Remote Sensing Analyst | `environmental_statistics` | Produces descriptive statistics and summaries of land cover and vegetation health, enabling the analyst to identify environmental patterns and trends. |
| Environmental Assessment Report Writer | No tools | Focuses on communicating the analyst's findings in a clear, stakeholder-friendly report without directly processing the dataset. |

Assigning only role-appropriate tools encourages specialization, minimizes unnecessary tool usage, and reflects how responsibilities are distributed among environmental data professionals in real-world projects.

# Task 3 – Define Tasks & Process

The workflow is implemented as a sequential CrewAI process.

The Environmental Data Quality Specialist first inspects the raw dataset and produces a structured quality assessment. The Remote Sensing Analyst then uses that assessment as context while interpreting the environmental data. Finally, the Environmental Assessment Report Writer uses the analysis to produce a stakeholder-ready report.

The sequential process therefore demonstrates how the output of one specialized agent becomes context for the next.

## Creating a Sample Environmental Monitoring Dataset

To demonstrate the CrewAI workflow without relying on external files, we generate a small synthetic environmental monitoring dataset.

The dataset represents observations exported from a remote sensing/GIS workflow and contains:

- NDVI (Normalized Difference Vegetation Index)
- Land Cover
- Rainfall
- Surface Temperature
- Vegetation Health

A few intentional data quality issues are introduced so that the Environmental Data Quality Specialist has meaningful work to perform before the analysis begins.

In [11]:
import pandas as pd

environmental_data = pd.DataFrame({
    "Plot_ID": [
        "P001","P002","P003","P004","P005",
        "P006","P007","P008","P009","P010",
        "P011","P012"
    ],

    "District": [
        "Faisalabad","Chiniot","Jhang","Toba Tek Singh",
        "Faisalabad","Chiniot","Jhang","Faisalabad",
        "Toba Tek Singh","Jhang","Chiniot","Faisalabad"
    ],

    "Latitude":[
        31.41,31.72,31.27,30.97,
        31.44,31.69,31.20,31.40,
        30.95,31.25,31.71,31.39
    ],

    "Longitude":[
        73.08,72.98,72.33,72.48,
        73.05,72.95,72.30,73.11,
        72.51,72.37,72.99,73.02
    ],

    "NDVI":[
        0.82,
        0.41,
        0.77,
        None,      # Missing value
        0.65,
        0.35,
        1.25,      # Invalid NDVI
        0.71,
        0.48,
        0.81,
        0.29,
        0.76
    ],

    "Land_Cover":[
        "Agriculture",
        "Barren",
        "Forest",
        "Agriculture",
        "Urban",
        "Barren",
        "Forest",
        "Agriculture",
        "Urban",
        "Forest",
        "Barren",
        "Agriculture"
    ],

    "Rainfall_mm":[
        102,
        45,
        118,
        93,
        None,      # Missing value
        38,
        122,
        110,
        65,
        119,
        40,
        108
    ],

    "Surface_Temp_C":[
        28.2,
        35.7,
        26.8,
        30.1,
        33.4,
        36.1,
        25.9,
        28.5,
        32.6,
        26.3,
        35.5,
        27.8
    ],

    "Vegetation_Health":[
        "Healthy",
        "Poor",
        "Excellent",
        "Healthy",
        "Moderate",
        "Poor",
        "Excellent",
        "Healthy",
        "Moderate",
        "Excellent",
        "Poor",
        "Healthy"
    ]
})

# Add one duplicate row intentionally
environmental_data = pd.concat(
    [environmental_data, environmental_data.iloc[[2]]],
    ignore_index=True
)

environmental_data.to_csv("environmental_monitoring_dataset.csv", index=False)

environmental_data

,Plot_ID,District,Latitude,Longitude,NDVI,Land_Cover,Rainfall_mm,Surface_Temp_C,Vegetation_Health
0,P001,Faisalabad,31.41,73.08,0.82,Agriculture,102.0,28.2,Healthy
1,P002,Chiniot,31.72,72.98,0.41,Barren,45.0,35.7,Poor
2,P003,Jhang,31.27,72.33,0.77,Forest,118.0,26.8,Excellent
3,P004,Toba Tek Singh,30.97,72.48,NaN,Agriculture,93.0,30.1,Healthy
4,P005,Faisalabad,31.44,73.05,0.65,Urban,NaN,33.4,Moderate
5,P006,Chiniot,31.69,72.95,0.35,Barren,38.0,36.1,Poor
6,P007,Jhang,31.20,72.30,1.25,Forest,122.0,25.9,Excellent
7,P008,Faisalabad,31.40,73.11,0.71,Agriculture,110.0,28.5,Healthy
8,P009,Toba Tek Singh,30.95,72.51,0.48,Urban,65.0,32.6,Moderate
9,P010,Jhang,31.25,72.37,0.81,Forest,119.0,26.3,Excellent


In [12]:
# validation

print("Dataset shape:", environmental_data.shape)
print("\nMissing values:")
print(environmental_data.isnull().sum())

print("\nDuplicate rows:", environmental_data.duplicated().sum())

print("\nNDVI range:")
print(environmental_data["NDVI"].min(), "to", environmental_data["NDVI"].max())

Dataset shape: (13, 9)

Missing values:
Plot_ID              0
District             0
Latitude             0
Longitude            0
NDVI                 1
Land_Cover           0
Rainfall_mm          1
Surface_Temp_C       0
Vegetation_Health    0
dtype: int64

Duplicate rows: 1

NDVI range:
0.29 to 1.25


## Task Definitions

The workflow consists of three dependent tasks.

Each task consumes the output of the previous task, demonstrating collaborative problem solving within a CrewAI workflow.

In [13]:
# Task 1
from crewai import Task

quality_check_task = Task(
    description="""
Use the dataset_quality_report tool to inspect:

environmental_monitoring_dataset.csv

Identify:

- missing values
- duplicate records
- NDVI values outside the valid range of -1 to 1
- negative rainfall values
- any other obvious data quality concerns

Do not modify, delete, or impute any records.

Only inspect the dataset and report the detected issues.
""",

    expected_output="""
A structured Markdown data quality report containing exactly these sections:

1. Dataset Overview
2. Missing Values
3. Duplicate Records
4. Invalid Environmental Measurements
5. Other Data Quality Issues
6. Recommendations for Analysis

Clearly identify the affected columns and, where possible, the affected Plot_ID values.
""",

    agent=environmental_data_quality_specialist
)

In [14]:
# Task 2
analysis_task = Task(
    description="""
Analyze the environmental monitoring dataset using the
environmental_statistics tool.

Use the Data Quality Specialist's assessment as context when
interpreting the results.

Analyze:

- NDVI patterns
- vegetation health distribution
- land cover distribution
- rainfall patterns
- surface temperature
- notable environmental relationships or trends
- areas that may require further monitoring

Do not treat invalid or missing observations as reliable evidence.
Clearly distinguish observed patterns from limitations caused by
data quality issues.
""",

    expected_output="""
A structured Markdown remote sensing analysis containing:

1. Key Statistics
2. NDVI and Vegetation Health Findings
3. Land Cover Findings
4. Rainfall and Environmental Patterns
5. Areas of Concern
6. Data Limitations
7. Recommendations

Use specific values or Plot_IDs where appropriate.
""",

    context=[quality_check_task],

    agent=remote_sensing_analyst
)

In [15]:
# Task 3
report_task = Task(
    description="""
Prepare a concise environmental monitoring report for
decision-makers using the Remote Sensing Analyst's findings.

The report should:

- summarize the most important environmental findings
- mention important data quality limitations
- identify areas requiring attention
- provide practical monitoring recommendations
- avoid introducing claims that are not supported by the analysis

Write for a stakeholder who understands environmental monitoring
but does not need to see the underlying technical process.
""",

    expected_output="""
A professional Markdown environmental monitoring report containing:

# Executive Summary

# Key Findings

# Environmental Concerns

# Data Limitations

# Recommendations

# Conclusion

The report should be concise, evidence-based, and stakeholder-friendly.
""",

    context=[analysis_task],

    agent=environmental_report_writer
)

## Assemble the Crew

The crew is configured using a **sequential process**, meaning each task executes in a predefined order.

The output produced by one agent becomes the context for the next agent, allowing the team to collaboratively solve the overall problem.

In [18]:
from crewai import Crew, Process

sequential_crew = Crew(
    agents=[
        environmental_data_quality_specialist,
        remote_sensing_analyst,
        environmental_report_writer
    ],

    tasks=[
        quality_check_task,
        analysis_task,
        report_task
    ],

    process=Process.sequential,

    # verbose=True
    verbose=False
)

In [19]:
sequential_result = sequential_crew.kickoff()

# Agent: Environmental Data Quality Specialist
## Task: 
Use the dataset_quality_report tool to inspect:

environmental_monitoring_dataset.csv

Identify:

- missing values
- duplicate records
- NDVI values outside the valid range of -1 to 1
- negative rainfall values
- any other obvious data quality concerns

Do not modify, delete, or impute any records.

Only inspect the dataset and report the detected issues.



# Agent: Environmental Data Quality Specialist
## Thought: We need to output the tool call.Thought: I need to generate a data quality report for the specified dataset file to identify missing values, duplicates, invalid NDVI and rainfall values, and any other issues.
## Using tool: dataset_quality_report
## Tool Input: 
"{\"file_path\": \"environmental_monitoring_dataset.csv\"}"
## Tool Output: 
=== DATASET OVERVIEW ===
Rows: 13
Columns: 9

Columns: ['Plot_ID', 'District', 'Latitude', 'Longitude', 'NDVI', 'Land_Cover', 'Rainfall_mm', 'Surface_Temp_C', 'Vegetation_Health']

==

## Output Format Refinement

During the sequential run, the handoff between the Data Quality Specialist and the Remote Sensing Analyst was reviewed.

The initial task output was not sufficiently structured for reliable downstream consumption. To improve the handoff, the `expected_output` of the quality assessment task was refined to require clearly labelled sections for missing values, duplicate records, invalid environmental measurements, other issues, and recommendations.

This makes the intermediate result more predictable and gives the Remote Sensing Analyst a consistent structure to interpret.

# Task 4 – Hierarchical Delegation

The same environmental monitoring problem is rebuilt using CrewAI's hierarchical process.

Instead of manually defining a fixed sequence of agent handoffs, a manager agent coordinates the specialized agents, delegates work, and reviews their results.

This allows us to compare a predictable sequential workflow with a more autonomous management-based workflow.

In the hierarchical version, a manager agent coordinates the three specialist agents.

Instead of explicitly defining the execution order, the manager is responsible for delegating work to the appropriate specialists, reviewing their outputs, and ensuring that the final environmental assessment satisfies the overall objective.

The same environmental monitoring problem is used so that the hierarchical approach can be compared fairly with the sequential crew.

In [31]:
# Manager Agent

manager_agent = Agent(
    role="Environmental Assessment Manager",

    goal=(
        "Coordinate the environmental monitoring analysis by delegating "
        "data quality inspection, remote sensing analysis, and report "
        "writing to the appropriate specialists. Review their work and "
        "ensure that the final assessment is accurate, complete, and "
        "suitable for environmental stakeholders."
    ),

    backstory=(
        "You are an experienced environmental project manager who oversees "
        "remote sensing and GIS-based environmental assessments. You are "
        "skilled at assigning work to specialists, reviewing analytical "
        "results, identifying inconsistencies, and ensuring that final "
        "reports are supported by the available evidence."
    ),

    llm=llm,

    allow_delegation=True,

    verbose=True
)

In [32]:
hierarchical_task = Task(
    description="""
    Produce a stakeholder-ready environmental monitoring assessment
    using the dataset:

    environmental_monitoring_dataset.csv

    Coordinate the following responsibilities through the available
    specialist agents:

    1. Inspect the dataset for missing values, duplicate records,
       invalid NDVI values, negative rainfall values, and other
       obvious data-quality issues.

    2. Analyse the environmental data, including NDVI, vegetation
       health, land-cover distribution, rainfall, surface temperature,
       and notable environmental patterns.

    3. Prepare a concise environmental assessment for stakeholders
       that summarizes the findings, limitations, areas requiring
       attention, and practical monitoring recommendations.

    Ensure that the final assessment does not introduce unsupported
    claims and clearly distinguishes observed findings from limitations.
    """,

    expected_output="""
    A stakeholder-ready environmental monitoring assessment containing:

    1. Executive Summary
    2. Key Environmental Findings
    3. Data Quality Limitations
    4. Areas Requiring Attention
    5. Monitoring Recommendations
    6. Conclusion

    All findings must be grounded in the available dataset and specialist
    analysis.
    """,

    agent=manager_agent
)

In [33]:
# build the hierarchical crew

hierarchical_crew = Crew(
    agents=[
        environmental_data_quality_specialist,
        remote_sensing_analyst,
        environmental_report_writer
    ],

    tasks=[
        hierarchical_task
    ],

    process=Process.hierarchical,

    manager_agent=manager_agent,

    # verbose=True
    verbose=False
)

In [34]:
hierarchical_result = await hierarchical_crew.kickoff_async()

# Agent: Environmental Assessment Manager
## Task: 
    Produce a stakeholder-ready environmental monitoring assessment
    using the dataset:

    environmental_monitoring_dataset.csv

    Coordinate the following responsibilities through the available
    specialist agents:

    1. Inspect the dataset for missing values, duplicate records,
       invalid NDVI values, negative rainfall values, and other
       obvious data-quality issues.

    2. Analyse the environmental data, including NDVI, vegetation
       health, land-cover distribution, rainfall, surface temperature,
       and notable environmental patterns.

    3. Prepare a concise environmental assessment for stakeholders
       that summarizes the findings, limitations, areas requiring
       attention, and practical monitoring recommendations.

    Ensure that the final assessment does not introduce unsupported
    claims and clearly distinguishes observed findings from limitations.
    
 

I encountered an error while tr

## Sequential vs Hierarchical Comparison

The same environmental monitoring problem was evaluated using two CrewAI execution strategies. The sequential crew follows a fixed pipeline from data quality inspection to remote sensing analysis and finally report writing. The hierarchical crew uses a manager agent to delegate work to the specialist agents and review their outputs.

| Aspect | Sequential CrewAI | Hierarchical CrewAI |
|---|---|---|
| Workflow | Fixed specialist sequence | Manager-directed delegation |
| Coordination | Explicit task dependencies | Manager delegates and reviews |
| Predictability | High | Lower |
| Flexibility | Lower | Higher |
| Complexity | Lower | Higher |
| Latency | 8.07 seconds | Not completed successfully |
| Token Usage | 21,326 total tokens | 40,593 total tokens |
| Successful Requests | 9 | 15 |
| Output Quality | Complete environmental assessment | No final assessment produced |
| Reliability | Successful execution | Failed during manager delegation |
| Cost | Lower relative token usage | Higher relative token usage |
| Best Use | Stable, well-defined workflows | Complex workflows requiring dynamic delegation |

### Interpretation

The sequential approach performed better for this particular task because the workflow has a clear dependency order: data quality inspection → environmental analysis → report writing. The hierarchical approach used substantially more tokens and requests but did not produce a final result because the manager's delegation call failed validation. Therefore, although hierarchical delegation provides greater flexibility, its additional complexity and delegation overhead were not justified for this predictable workflow.

# Task 5 – Evaluation & Cost Awareness

The three approaches are evaluated using:

1. Factual grounding
2. Completeness
3. Stakeholder quality

Latency and token usage are also considered where execution information is available.

In [35]:
# success criteria

success_criteria = {
    "Factual Grounding": (
        "The output accurately reflects the dataset and does not introduce "
        "unsupported environmental claims."
    ),

    "Completeness": (
        "The output covers important findings, data-quality limitations, "
        "areas requiring attention, and monitoring recommendations."
    ),

    "Stakeholder Quality": (
        "The final report is clear, organised, professional, and understandable "
        "to an environmental stakeholder."
    )
}

for criterion, description in success_criteria.items():
    print(f"{criterion}:")
    print(description)
    print()


Factual Grounding:
The output accurately reflects the dataset and does not introduce unsupported environmental claims.

Completeness:
The output covers important findings, data-quality limitations, areas requiring attention, and monitoring recommendations.

Stakeholder Quality:
The final report is clear, organised, professional, and understandable to an environmental stakeholder.



In [36]:
# manual evaluation

evaluation_scores = {
    "Sequential Crew": {
        "Factual Grounding": 4,
        "Completeness": 5,
        "Stakeholder Quality": 5
    }
}

evaluation_scores

{'Sequential Crew': {'Factual Grounding': 4,
  'Completeness': 5,
  'Stakeholder Quality': 5}}

In [38]:
evaluation_scores["Hierarchical Crew"] = {
    "Factual Grounding": 4,
    "Completeness": 4,
    "Stakeholder Quality": 4
}

evaluation_scores

{'Sequential Crew': {'Factual Grounding': 4,
  'Completeness': 5,
  'Stakeholder Quality': 5},
 'Hierarchical Crew': {'Factual Grounding': 4,
  'Completeness': 4,
  'Stakeholder Quality': 4}}

In [27]:
# creating a simple timing helper

import time

def run_and_measure(crew):
    start_time = time.perf_counter()

    result = crew.kickoff()

    elapsed = time.perf_counter() - start_time

    return result, elapsed

In [28]:
sequential_result, sequential_time = run_and_measure(
    sequential_crew
)

print(f"Sequential latency: {sequential_time:.2f} seconds")

# Agent: Environmental Data Quality Specialist
## Task: 
Use the dataset_quality_report tool to inspect:

environmental_monitoring_dataset.csv

Identify:

- missing values
- duplicate records
- NDVI values outside the valid range of -1 to 1
- negative rainfall values
- any other obvious data quality concerns

Do not modify, delete, or impute any records.

Only inspect the dataset and report the detected issues.



# Agent: Environmental Data Quality Specialist
## Thought: We need to output the tool call.Thought: I need to generate a data quality report for the specified dataset file to identify missing values, duplicates, invalid NDVI and rainfall values, and any other issues.
## Using tool: dataset_quality_report
## Tool Input: 
"{\"file_path\": \"environmental_monitoring_dataset.csv\"}"
## Tool Output: 
=== DATASET OVERVIEW ===
Rows: 13
Columns: 9

Columns: ['Plot_ID', 'District', 'Latitude', 'Longitude', 'NDVI', 'Land_Cover', 'Rainfall_mm', 'Surface_Temp_C', 'Vegetation_Health']

==

In [29]:
hierarchical_result, hierarchical_time = run_and_measure(
    hierarchical_crew
)

print(f"Hierarchical latency: {hierarchical_time:.2f} seconds")

# Agent: Environmental Monitoring Project Manager
## Task: 
Complete a full environmental monitoring assessment using:

environmental_monitoring_dataset.csv

Coordinate the following specialists:

1. Environmental Data Quality Specialist
   - Inspect the dataset
   - Identify missing values, duplicates, invalid NDVI values,
     and other data quality problems

2. Remote Sensing Analyst
   - Analyze NDVI, vegetation health, land cover, rainfall,
     and surface temperature
   - Identify environmental patterns and areas of concern

3. Environmental Assessment Report Writer
   - Convert the validated analysis into a concise,
     stakeholder-ready environmental report

Review the specialists' work and ensure that the final result:

- is factually grounded in the dataset
- acknowledges data quality limitations
- contains meaningful environmental insights
- provides useful recommendations
- follows a clear professional structure

 Received None or empty response from LLM call.
 An unknown

In [30]:
# Inspect CrewAi output metrics

print(type(sequential_result))
print(dir(sequential_result))

<class 'crewai.crews.crew_output.CrewOutput'>
['__abstractmethods__', '__annotations__', '__class__', '__class_getitem__', '__class_vars__', '__copy__', '__deepcopy__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__fields__', '__fields_set__', '__format__', '__ge__', '__get_pydantic_core_schema__', '__get_pydantic_json_schema__', '__getattr__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__pretty__', '__private_attributes__', '__pydantic_complete__', '__pydantic_computed_fields__', '__pydantic_core_schema__', '__pydantic_custom_init__', '__pydantic_decorators__', '__pydantic_extra__', '__pydantic_extra_info__', '__pydantic_fields__', '__pydantic_fields_set__', '__pydantic_generic_metadata__', '__pydantic_init_subclass__', '__pydantic_on_complete__', '__pydantic_parent_namespace__', '__pydantic_post_init__', '__pydantic_private__', '__pydantic

In [39]:
print(type(hierarchical_result))
print(dir(hierarchical_result))

<class 'crewai.crews.crew_output.CrewOutput'>
['__abstractmethods__', '__annotations__', '__class__', '__class_getitem__', '__class_vars__', '__copy__', '__deepcopy__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__fields__', '__fields_set__', '__format__', '__ge__', '__get_pydantic_core_schema__', '__get_pydantic_json_schema__', '__getattr__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__pretty__', '__private_attributes__', '__pydantic_complete__', '__pydantic_computed_fields__', '__pydantic_core_schema__', '__pydantic_custom_init__', '__pydantic_decorators__', '__pydantic_extra__', '__pydantic_extra_info__', '__pydantic_fields__', '__pydantic_fields_set__', '__pydantic_generic_metadata__', '__pydantic_init_subclass__', '__pydantic_on_complete__', '__pydantic_parent_namespace__', '__pydantic_post_init__', '__pydantic_private__', '__pydantic

In [40]:
print(sequential_result.token_usage)

total_tokens=21326 prompt_tokens=11467 cached_prompt_tokens=1664 completion_tokens=9859 successful_requests=9


In [41]:
print(hierarchical_result.token_usage)

total_tokens=40593 prompt_tokens=23355 cached_prompt_tokens=3328 completion_tokens=17238 successful_requests=15


## Evaluation Criteria

Each approach is evaluated using three criteria, scored from 1 to 5:

| Criterion | Description |
|---|---|
| Factual Grounding | Findings accurately reflect the available dataset and avoid unsupported claims. |
| Completeness | The output covers the important findings, data-quality limitations, areas requiring attention, and recommendations. |
| Stakeholder Quality | The output is clear, organised, professional, and understandable to environmental stakeholders. |

Maximum score per completed run: **15 points**.

## Manual Evaluation

| Run | Approach | Factual Grounding /5 | Completeness /5 | Stakeholder Quality /5 | Total /15 |
|---|---|---:|---:|---:|---:|
| 1 | Sequential CrewAI | 4 | 5 | 5 | 14 |
| 2 | Hierarchical CrewAI | N/A | N/A | N/A | N/A |
| 3 | Single-Agent LangGraph | See Day 3 results | See Day 3 results | See Day 3 results | See Day 3 results |

### Evaluation Notes

The sequential CrewAI run received a score of **14/15**. It produced a complete and professionally structured environmental assessment, although some claims in the generated report went beyond the information explicitly returned by the data-quality tool, so factual grounding was scored 4 rather than 5.

The hierarchical run was not assigned a quality score because it failed during manager-level delegation and did not produce a completed final assessment. This failure is instead recorded as part of the reliability comparison.

The Day 3 single-agent LangGraph result should be evaluated using the results already recorded in the Day 3 notebook rather than rerunning it.

In [42]:
# reliability

reliability_comparison = {
    "Sequential CrewAI": "Successful execution",
    "Hierarchical CrewAI": "Successful or failed, based on observed run",
    "Day 3 LangGraph": "Based on Day 3 execution"
}

reliability_comparison

{'Sequential CrewAI': 'Successful execution',
 'Hierarchical CrewAI': 'Successful or failed, based on observed run',
 'Day 3 LangGraph': 'Based on Day 3 execution'}

## Final Comparison

| Approach | Quality | Latency | Token Usage | Reliability | Complexity |
|---|---|---:|---:|---|---|
| Sequential CrewAI | 14/15 | 8.07 s | 21,326 | Successful | Medium |
| Hierarchical CrewAI | Not scored due to failed execution | Not available | 40,593 | Failed during manager delegation | High |
| Single-Agent LangGraph | See Day 3 evaluation | See Day 3 results | See Day 3 results | See Day 3 results | Low–Medium |

### Key Observation

The sequential CrewAI workflow produced a complete result with fewer tokens and fewer successful requests than the hierarchical attempt. The hierarchical approach consumed approximately **1.9× as many total tokens** as the sequential run, while failing to produce a final report. This suggests that, for this particular predictable environmental monitoring workflow, the additional coordination overhead of hierarchical delegation did not provide a practical advantage.

## Was Multi-Agent Collaboration Worth It?

For this environmental monitoring task, the multi-agent approach was useful because data-quality inspection, environmental analysis, and stakeholder reporting are distinct responsibilities that can be naturally separated among specialists. The sequential CrewAI workflow successfully produced a complete environmental assessment and achieved a strong manual score of 14/15, demonstrating the benefit of specialization. However, the hierarchical approach introduced substantially more token usage and delegation complexity without producing a successful final result. For this specific task, the sequential multi-agent approach was worth the added complexity compared with a single generalist agent, while hierarchical delegation was not justified because the workflow was already predictable and well-defined.